In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, optimizers, callbacks

from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

MAIN_DIR = "/kaggle/input/v000ct1/0000"
OUTPUT_DIR = "dataset_split"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 50
INITIAL_LR = 0.0001
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

In [ ]:
def split_dataset(source_dir, output_dir, train_size=0.70, val_size=0.15, test_size=0.15, seed=42):
    assert abs(train_size + val_size + test_size - 1.0) < 0.01
    
    train_dir = os.path.join(output_dir, 'train')
    val_dir = os.path.join(output_dir, 'validation')
    test_dir = os.path.join(output_dir, 'test')
    
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    classes = sorted([d for d in os.listdir(source_dir) 
                     if os.path.isdir(os.path.join(source_dir, d))])
    
    for class_name in classes:
        class_path = os.path.join(source_dir, class_name)
        
        images = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            images.extend(Path(class_path).glob(ext))
            images.extend(Path(class_path).glob(ext.upper()))
        
        images = [str(img) for img in images]
        if len(images) == 0:
            continue
        
        train_val_imgs, test_imgs = train_test_split(
            images, test_size=test_size, random_state=seed, shuffle=True
        )
        
        val_size_adjusted = val_size / (train_size + val_size)
        train_imgs, val_imgs = train_test_split(
            train_val_imgs, test_size=val_size_adjusted, random_state=seed, shuffle=True
        )
        
        train_class_dir = os.path.join(train_dir, class_name)
        val_class_dir = os.path.join(val_dir, class_name)
        test_class_dir = os.path.join(test_dir, class_name)
        
        os.makedirs(train_class_dir, exist_ok=True)
        os.makedirs(val_class_dir, exist_ok=True)
        os.makedirs(test_class_dir, exist_ok=True)
        
        for img_path in train_imgs:
            shutil.copy2(img_path, os.path.join(train_class_dir, os.path.basename(img_path)))
        
        for img_path in val_imgs:
            shutil.copy2(img_path, os.path.join(val_class_dir, os.path.basename(img_path)))
        
        for img_path in test_imgs:
            shutil.copy2(img_path, os.path.join(test_class_dir, os.path.basename(img_path)))
    
    return train_dir, val_dir, test_dir

print("Splitt dataset")
TRAIN_DIR, VAL_DIR, TEST_DIR = split_dataset(
    MAIN_DIR, OUTPUT_DIR, 
    train_size=TRAIN_SPLIT, 
    val_size=VAL_SPLIT, 
    test_size=TEST_SPLIT
)
print("Dataset split complete")

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.25,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"Classes: {class_names}\n")

In [6]:
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

In [ ]:
def build_model():
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    
    base_model.trainable = True
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu', 
                     kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='TumorClassifier')
    
    return model

model = build_model()

model.compile(
    optimizer=optimizers.Adam(learning_rate=INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

print(f"Model built. Parameters: {model.count_params():,}")

In [ ]:
callbacks_list = [
    callbacks.ModelCheckpoint(
        'best_tumor_classifier.keras',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=5,
        min_lr=0.0000001,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

In [ ]:
print("TRAINING")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    class_weight=class_weight_dict,
    verbose=1
)

print("Done+++")

In [ ]:
print("TEST")
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

print("Classification rfeport test:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
fig1, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
epochs_range = range(1, len(history.history['accuracy']) + 1)

# 1. Accuracy
axes[0].plot(epochs_range, history.history['accuracy'], 'o-', label='Train', markersize=4, color='blue')
axes[0].plot(epochs_range, history.history['val_accuracy'], 's-', label='Validation', markersize=4, color='orange')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold', pad=10)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)
axes[0].set_ylim([0, 1.05])

# 2. Loss
axes[1].plot(epochs_range, history.history['loss'], 'o-', label='Train', markersize=4, color='blue')
axes[1].plot(epochs_range, history.history['val_loss'], 's-', label='Validation', markersize=4, color='orange')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold', pad=10)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

# 3. Precision
axes[2].plot(epochs_range, history.history['precision'], 'o-', label='Train', markersize=4, color='blue')
axes[2].plot(epochs_range, history.history['val_precision'], 's-', label='Validation', markersize=4, color='orange')
axes[2].set_title('Model Precision', fontsize=14, fontweight='bold', pad=10)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Precision', fontsize=12)
axes[2].legend(fontsize=11)
axes[2].grid(alpha=0.3)
axes[2].set_ylim([0, 1.05])

# 4. Recall
axes[3].plot(epochs_range, history.history['recall'], 'o-', label='Train', markersize=4, color='blue')
axes[3].plot(epochs_range, history.history['val_recall'], 's-', label='Validation', markersize=4, color='orange')
axes[3].set_title('Model Recall', fontsize=14, fontweight='bold', pad=10)
axes[3].set_xlabel('Epoch', fontsize=12)
axes[3].set_ylabel('Recall', fontsize=12)
axes[3].legend(fontsize=11)
axes[3].grid(alpha=0.3)
axes[3].set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# C-Matrix
fig2, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={'size': 12}, ax=ax,
            cbar_kws={'label': 'Percentage'})

ax.set_title('Normalized Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('True Label', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=14, fontweight='bold')

ax.tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
final_test_acc = model.evaluate(test_generator, verbose=0)[1]
best_val_acc = max(history.history['val_accuracy'])
best_epoch = np.argmax(history.history['val_accuracy']) + 1

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
print("RESULTS")
print(f"Best Model (Epoch {best_epoch}):")
print(f"Validation Accuracy: {best_val_acc*100:.2f}%")
print(f"Test Accuracy: {final_test_acc*100:.2f}%")
print(f"Test Precision: {precision*100:.2f}%")
print(f"Test Recall: {recall*100:.2f}%")
print(f"Test F1-Score: {f1*100:.2f}%")

print("Saved")
model.save('final_tumor_classifier.keras')
